In [ ]:
# ── Colab Setup ───────────────────────────────────────────────────────────────
# Run this cell first every time you open a new Colab session.
# Clones the repo (data and artifacts included) and configures the environment.
import os, sys

REPO_URL  = "https://github.com/tackes/Modern-Time-Series-Forecasting-Cohort.git"
REPO_PATH = "/content/packt-modern-time-series"

if not os.path.exists(REPO_PATH):
    os.system(f"git clone -q {REPO_URL} {REPO_PATH}")

# Stay in student_notebooks so Path().resolve().parent resolves to repo root
os.chdir(f"{REPO_PATH}/student_notebooks")

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

print(f"✓ Setup complete — {os.getcwd()}")

# Module 7 — Prediction Intervals
**Type:** [Watch Only / Light Execution]  
**Time:** 25 minutes  
**Job:** Judge uncertainty quality across all models. Score 80% intervals. Visualize fan charts. Build the uncertainty leaderboard.

---
## 7.1 — Setup
**[Watch Only]**

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings("ignore")

matplotlib.rcParams['figure.figsize'] = (14, 4)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

from config import (
    ARTIFACT_DIR,
    WORKSHOP_SUBSET_N,
    MICRO_SUBSET_N,
    INTERVAL_COVERAGE,
)
from src.checkpointing import load_checkpoint
from src.evaluation import score_forecasts, pooled_interval_score, coverage_rate
from src.schemas import validate_score

print("Setup complete.")

---
## 7.2 — Why Intervals Matter
**[Watch Only]**

Point forecasts are incomplete. A model that predicts 100 units cannot tell you how much safety stock to hold.

**The procurement scenario:**  
Your model predicts 100 units for next month. The 80% prediction interval is [90, 110]. You hold 10 units of safety stock. Reasonable.

Same model, same prediction: 100 units. The 80% interval is [10, 500]. You now need 400 units of safety stock to cover the upside — or you accept a 20% stockout risk. That forecast is useless for a purchasing decision.

**wMAPE does not surface this problem.** Two models with identical wMAPE can have wildly different interval quality. A model with narrow, well-calibrated intervals is operationally superior to one with identical point accuracy and erratic intervals.

**Interval Score penalizes both dimensions simultaneously:**
- Wide intervals: penalized by the width term regardless of coverage
- Misses: penalized at 10× the miss size (for 80% intervals)

This module scores every model on interval quality. The uncertainty leaderboard may look different from the wMAPE leaderboard. That difference is the information you need for a production decision.

---
## 7.3 — Load All Forecast Artifacts
**[Watch Only]**

In [ ]:
baseline_full = load_checkpoint("04_baseline_forecasts")
ml_full       = load_checkpoint("05_ml_forecasts")
dl_full       = load_checkpoint("06_dl_forecasts")

all_forecasts = pd.concat([baseline_full, ml_full, dl_full], ignore_index=True)

models_with_intervals = [
    m for m in all_forecasts["model"].unique()
    if all_forecasts[all_forecasts["model"] == m][["lo_80", "hi_80"]].notna().all(axis=None)
]

print(f"Total forecast rows  : {len(all_forecasts):,}")
print(f"Models in panel      : {sorted(all_forecasts['model'].unique().tolist())}")
print(f"Models with intervals: {sorted(models_with_intervals)}")

**Expected output:**
```
Total forecast rows  : 504,000
Models in panel      : ['AutoETS', 'Chronos-t5-mini', 'LightGBM', 'NHITS', 'Naive', 'SeasonalNaive']
Models with intervals: ['AutoETS', 'Chronos-t5-mini', 'LightGBM', 'NHITS', 'Naive', 'SeasonalNaive']
```

---
## 7.4 — Fan Chart: One Series, All Models
**[Watch Only]**

In [ ]:
panel = load_checkpoint("03_validated_panel")

sample_uid = (
    panel.groupby("unique_id")["y"].sum()
    .sort_values(ascending=False)
    .index[0]
)
sample_cut = all_forecasts["cutoff"].max()
plot_start = sample_cut - pd.Timedelta(days=42)
actuals    = panel[panel["unique_id"] == sample_uid].set_index("ds")["y"]

model_styles = {
    "Naive":           {"color": "#E53935", "ls": "--"},
    "SeasonalNaive":   {"color": "#1E88E5", "ls": "--"},
    "AutoETS":         {"color": "#43A047", "ls": "--"},
    "Chronos-t5-mini": {"color": "#FB8C00", "ls": "-."},
    "LightGBM":        {"color": "#7B1FA2", "ls": "--"},
    "NHITS":           {"color": "#F4511E", "ls": "--"},
}

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.flatten()

for idx, model_name in enumerate(sorted(models_with_intervals)):
    ax = axes[idx]
    style = model_styles.get(model_name, {"color": "#555", "ls": "--"})
    act_plot = actuals[actuals.index >= plot_start]
    ax.plot(act_plot.index, act_plot.values,
            color="#333", linewidth=1.0, label="Actual", zorder=3)
    fcast = all_forecasts[
        (all_forecasts["unique_id"] == sample_uid) &
        (all_forecasts["cutoff"]    == sample_cut) &
        (all_forecasts["model"]     == model_name)
    ].set_index("ds")
    if len(fcast) > 0:
        ax.plot(fcast.index, fcast["y_hat"],
                color=style["color"], linewidth=1.5, linestyle=style["ls"],
                label=model_name, zorder=4)
        if "lo_80" in fcast.columns and fcast["lo_80"].notna().any():
            ax.fill_between(fcast.index, fcast["lo_80"], fcast["hi_80"],
                            alpha=0.18, color=style["color"])
    ax.axvline(sample_cut, color="#aaa", linestyle=":", linewidth=0.8)
    ax.set_title(model_name, fontsize=10)
    ax.set_ylabel("Units")
    ax.tick_params(axis="x", labelsize=7)

plt.suptitle(f"80% Prediction Intervals — {sample_uid} (Window 3)", fontsize=11, y=1.01)
plt.tight_layout()
plt.show()
print("Wider shaded band = less certain model. Narrower band that still covers actuals = better.")

**Expected output:** A 2×3 grid of fan charts, one per model. Each shows actuals, the point forecast, and the 80% interval as a shaded band.

---
## 7.5 — Interval Width Distribution
**[Watch Only]**

In [ ]:
interval_df = all_forecasts[
    all_forecasts["model"].isin(models_with_intervals) &
    all_forecasts["lo_80"].notna()
].copy()
interval_df["width"] = interval_df["hi_80"] - interval_df["lo_80"]

fig, ax = plt.subplots(figsize=(12, 4))
for model_name in sorted(models_with_intervals):
    style = model_styles.get(model_name, {"color": "#555"})
    widths = interval_df[interval_df["model"] == model_name]["width"]
    ax.hist(widths.clip(upper=widths.quantile(0.99)),
            bins=60, alpha=0.4, color=style["color"],
            label=model_name, density=True)
ax.set_xlabel("Interval width (hi_80 − lo_80)")
ax.set_ylabel("Density")
ax.set_title("Interval Width Distribution by Model (Full Panel, All Windows)", fontsize=10)
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

print("Median interval widths:")
medians = interval_df.groupby("model")["width"].median().sort_values()
for m, w in medians.items():
    print(f"  {m:<20}: {w:.1f} units")

**Expected output:** Overlapping histograms. Tighter distributions at lower widths = more confident intervals.

---
## 7.6 — Score All Intervals
**[Watch Only]**

In [ ]:
all_scores = score_forecasts(
    all_forecasts,
    subset_name=f"workshop_{WORKSHOP_SUBSET_N}",
)
all_scores = validate_score(all_scores, artifact_name="07_uncertainty_leaderboard")

uncertainty_lb = (
    all_scores
    .pivot_table(index="model", columns="metric", values="score")
    .reset_index()
)
uncertainty_lb.columns.name = None

if "IntervalScore_80" in uncertainty_lb.columns:
    uncertainty_lb = uncertainty_lb.sort_values("IntervalScore_80")

print("Uncertainty Leaderboard:")
print(uncertainty_lb.to_string(index=False))

**Expected output:**
```
Uncertainty Leaderboard:
          model  Coverage_80  IntervalScore_80    wMAPE
       LightGBM        0.XX          XX.XXXX   0.XXXX
          NHITS        0.XX          XX.XXXX   0.XXXX
        AutoETS        0.XX          XX.XXXX   0.XXXX
Chronos-t5-mini        0.XX          XX.XXXX   0.XXXX
  SeasonalNaive        0.XX          XX.XXXX   0.XXXX
          Naive        0.XX          XX.XXXX   0.XXXX
```

---
## 7.7 — Coverage Diagnostic
**[Watch Only]**

In [ ]:
if "Coverage_80" in uncertainty_lb.columns:
    cov_data = uncertainty_lb[["model", "Coverage_80"]].dropna().sort_values("Coverage_80", ascending=False)
    fig, ax = plt.subplots(figsize=(9, 4))
    bar_colors = [
        "#43A047" if abs(v - INTERVAL_COVERAGE) < 0.05
        else ("#FF9800" if v > INTERVAL_COVERAGE else "#E53935")
        for v in cov_data["Coverage_80"]
    ]
    bars = ax.barh(cov_data["model"], cov_data["Coverage_80"],
                   color=bar_colors, edgecolor="white")
    ax.axvline(INTERVAL_COVERAGE, color="#333", linestyle="--", linewidth=1.2,
               label=f"Target: {int(INTERVAL_COVERAGE*100)}%")
    ax.set_xlabel("Coverage (fraction of actuals inside 80% interval)")
    ax.set_title("Coverage Diagnostic — 80% Prediction Intervals", fontsize=10)
    ax.legend(fontsize=9)
    ax.invert_yaxis()
    for bar, val in zip(bars, cov_data["Coverage_80"]):
        ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
                f"{val:.1%}", va="center", fontsize=9)
    plt.tight_layout()
    plt.show()
    print("Green = within 5% of target | Orange = conservative | Red = overconfident")
    print("Coverage is diagnostic only — not used for ranking.")

**Expected output:** Horizontal bar chart showing coverage per model. Green = well-calibrated, orange = too wide, red = too narrow.

---
## 7.8 — wMAPE vs Interval Score Scatter
**[Watch Only]**

In [ ]:
if "IntervalScore_80" in uncertainty_lb.columns and "wMAPE" in uncertainty_lb.columns:
    fig, ax = plt.subplots(figsize=(8, 6))
    for _, row in uncertainty_lb.dropna(subset=["wMAPE", "IntervalScore_80"]).iterrows():
        style = model_styles.get(row["model"], {"color": "#555"})
        ax.scatter(row["wMAPE"], row["IntervalScore_80"],
                   color=style["color"], s=120, zorder=3)
        ax.annotate(row["model"],
                    xy=(row["wMAPE"], row["IntervalScore_80"]),
                    xytext=(6, 2), textcoords="offset points",
                    fontsize=8, color=style["color"])
    ax.set_xlabel("wMAPE (lower = better point accuracy)")
    ax.set_ylabel("Interval Score 80% (lower = better uncertainty)")
    ax.set_title("Point Accuracy vs Uncertainty Quality", fontsize=11)
    plt.tight_layout()
    plt.show()
    print("Bottom-left = best on both dimensions.")
    print("A model can win wMAPE but lose on interval quality — and vice versa.")

**Expected output:** Scatter plot with one point per model. Models toward the bottom-left dominate on both accuracy and uncertainty.

---
## 7.9 — Save the Uncertainty Leaderboard Artifact
**[Watch Only]**

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
uncertainty_path = ARTIFACT_DIR / "07_uncertainty_leaderboard.parquet"
all_scores.to_parquet(uncertainty_path, index=False)
print(f"  ✓ Uncertainty leaderboard saved: {uncertainty_path.name} ({len(all_scores):,} rows)")

**Expected output:**
```
  ✓ Uncertainty leaderboard saved: 07_uncertainty_leaderboard.parquet (18 rows)
```

---
## 7.10 — The Enterprise Cliff
**[Watch Only]**

We scored intervals using in-sample residuals as the calibration source. This is the right approach for a workshop. In production it introduces three problems worth knowing:

**Residual distribution shift:**  
In-sample residuals reflect the training period. If demand volatility increases post-deployment — a new competitor, a supply shock — the intervals become systematically too narrow. Production interval calibration requires a continuously updated holdout set.

**Per-series vs pooled calibration:**  
We pooled residuals across all 1,000 series to derive a single offset. High-volume stable SKUs have tighter residuals than low-volume intermittent ones. Pooled calibration overestimates uncertainty on the stable SKUs and underestimates it on the volatile ones. Enterprise systems maintain separate calibration per product category or volatility tier.

**Coverage is not the goal — decision quality is:**  
An 80% interval that is perfectly calibrated but centered on the wrong forecast still leads to the wrong purchasing decision. Coverage tells you about interval mechanics. It does not tell you whether the intervals are useful for the decision they are supposed to support.